### Import Libraries
Import necessary libraries for data handling, model training, and evaluation. Set up the device for GPU or CPU.



In [ ]:
# Import necessary libraries
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure CUDA is available for faster training
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


### Load Data Functions
Functions for loading labeled and unlabeled data from Excel files, converting them to tensors for model training.

### Load Data Files
Load the embeddings and labels for training, testing, and synthetic data using the defined functions.



In [ ]:
# Load labeled data with labels in the first column
def load_labeled_data(file_path, label_col=0):
    df = pd.read_excel(file_path, header=0)
    labels = df.iloc[:, label_col].values
    embeddings = df.iloc[:, 1:].values  # Embeddings start from the second column
    label_map = {'Racist': 0, 'Caste': 1, 'Offensive': 2, 'Sexist': 3}
    labels = torch.tensor([label_map[label] for label in labels], dtype=torch.long).to(device)
    embeddings = torch.tensor(embeddings, dtype=torch.float32).to(device)
    return embeddings, labels

# Load unlabeled data
def load_unlabeled_data(file_path):
    df = pd.read_excel(file_path, header=0)
    embeddings = torch.tensor(df.values, dtype=torch.float32).to(device)
    return embeddings

# Load the data files
train_embeddings, train_labels = load_labeled_data('Concatenated_Train_Embeddings.xlsx')
test_embeddings, test_labels = load_labeled_data('Concatenated_Test_Data.xlsx')
unlabeled_embeddings = load_unlabeled_data('Concatenated_Synthetic_Train_Data.xlsx')
synthetic_test_embeddings, synthetic_test_labels = load_labeled_data('Concatenated_Synthetic_Test_Data.xlsx')


### Model Definition
Defines the `MultimodalResNet` model based on ResNet-18, customized for multimodal input and adapted for a classification task with 4 output classes.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import torchvision.models as models

# Model Definition
class MultimodalResNet(nn.Module):
    def __init__(self, input_dim=1536, num_classes=4):
        super(MultimodalResNet, self).__init__()
        self.resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

        # Modifing the fully connected layers
        self.resnet.fc = nn.Sequential(
            nn.Linear(self.resnet.fc.in_features, 1024),
            nn.ReLU(),
            nn.BatchNorm1d(1024),
            nn.Dropout(0.5),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        batch_size = x.size(0)
        x = x.view(batch_size, 3, 32, 16)
        x = self.softmax(x)
        return x


### Training Function
The `train_model` function trains the given model using labeled data and optimizes it using backpropagation. It logs and returns the training loss history.

### Evaluation Function
The `evaluate_model` function evaluates the trained model's performance on a test dataset. It calculates and returns the accuracy, F1-score, a classification report, and a confusion matrix.


In [ ]:
def train_model(model, criterion, optimizer, train_loader, num_epochs=50):
    model.train()
    train_loss_history = []
    for epoch in range(num_epochs):
        running_loss = 0.0
        for inputs, labels in train_loader:
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        epoch_loss = running_loss / len(train_loader)
        train_loss_history.append(epoch_loss)
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}")
    return train_loss_history

# Evaluation Function
def evaluate_model(model, test_loader):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for inputs, labels in test_loader:
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    accuracy = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='weighted')
    report = classification_report(all_labels, all_preds, target_names=['Racist', 'Caste', 'Offensive', 'Sexist'])
    conf_matrix = confusion_matrix(all_labels, all_preds)
    return accuracy, f1, report, conf_matrix

### Teacher-Student Training Function
The `train_teacher_student_model` function implements the iterative training process of a teacher-student framework for semi-supervised learning. The function:

- Initializes and trains the teacher model on labeled data.
- Generates pseudo-labels for unlabeled data using the trained teacher model.
- Combines labeled and pseudo-labeled data for training the student model.
- Trains the student model with cross-entropy loss and KL divergence for consistency with the teacher's predictions.
- Promotes the student model to replace the teacher after each iteration.

#### Return Values:
- Trained `teacher_model` and `student_model`.
- Loss history for both teacher and student models across all epochs and iterations.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score, f1_score

# Training function without plotting loss curves and printing only final losses
def train_teacher_student_model(train_loader, unlabeled_embeddings, train_embeddings, train_labels, num_epochs=30, num_iterations=2):
    # Initialize teacher and student models
    teacher_model = MultimodalResNet().to(device)
    student_model = MultimodalResNet().to(device)

    # Cross-entropy loss for labeled data
    criterion = nn.CrossEntropyLoss()

    # Lists to store losses for teacher and student models
    all_train_loss_teacher = []
    all_train_loss_student = []

    for iteration in range(num_iterations):

        optimizer_teacher = optim.Adam(teacher_model.parameters(), lr=1e-4)
        train_loss_teacher = train_model(teacher_model, criterion, optimizer_teacher, train_loader, num_epochs)
        all_train_loss_teacher.append(train_loss_teacher)

        # Generate pseudo-labels for unlabeled data using the teacher model
        teacher_model.eval()
        with torch.no_grad():
            pseudo_labels = torch.argmax(teacher_model(unlabeled_embeddings), dim=1)

        # Create pseudo-labeled dataset for the student model
        pseudo_dataset = TensorDataset(unlabeled_embeddings, pseudo_labels)
        pseudo_loader = DataLoader(pseudo_dataset, batch_size=8, shuffle=True)

        # Combine labeled and pseudo-labeled data for the student model
        combined_train_loader = DataLoader(
            TensorDataset(
                torch.cat([train_embeddings, unlabeled_embeddings], dim=0),
                torch.cat([train_labels, pseudo_labels], dim=0)
            ),
            batch_size=8, shuffle=True
        )

        # Train the student model

        optimizer_student = optim.Adam(student_model.parameters(), lr=1e-4)
        student_model.train()

        train_loss_student = []
        for epoch in range(num_epochs):
            running_loss = 0.0
            for inputs, labels in combined_train_loader:
                optimizer_student.zero_grad()

                # Forward pass for labeled data
                outputs = student_model(inputs)
                loss_labeled = criterion(outputs, labels)

                # Compute KL divergence for pseudo-labeled data separately
                pseudo_inputs, _ = next(iter(pseudo_loader))
                student_outputs_unlabeled = student_model(pseudo_inputs)
                with torch.no_grad():
                    teacher_outputs_unlabeled = teacher_model(pseudo_inputs)

                kl_loss = F.kl_div(
                    F.log_softmax(student_outputs_unlabeled, dim=1),
                    F.softmax(teacher_outputs_unlabeled, dim=1),
                    reduction='batchmean'
                )

                # Total loss for student training
                total_loss = loss_labeled + kl_loss
                total_loss.backward()

                # Backward pass and optimization
                optimizer_student.step()
                running_loss += total_loss.item()

            # Calculate and store average student loss for the epoch
            epoch_loss = running_loss / len(combined_train_loader)
            train_loss_student.append(epoch_loss)


        # Store the student's loss per epoch for this iteration
        all_train_loss_student.append(train_loss_student)

        # Promote the student to be the new teacher for the next iteration
        teacher_model.load_state_dict(student_model.state_dict())

    print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {epoch_loss:.4f}")

    return teacher_model, student_model, all_train_loss_teacher, all_train_loss_student


### Data Preparation and Training
- **DataLoaders**: Created for training and test data with batch size 8.
- **Model Training**: `train_teacher_student_model()` runs for 50 epochs over 3 iterations.
- **Outputs**: Trained `teacher_model`, `student_model`, and their training loss histories.


In [ ]:
# Create DataLoader objects
train_dataset = TensorDataset(train_embeddings, train_labels)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)

test_dataset = TensorDataset(test_embeddings, test_labels)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

# Train the teacher and student models
teacher_model, student_model, all_train_loss_teacher, all_train_loss_student = train_teacher_student_model(train_loader, unlabeled_embeddings, train_embeddings, train_labels, num_epochs=50, num_iterations=3)


In [ ]:
import matplotlib.pyplot as plt

def plot_training_curve(train_loss_history, title):
    plt.figure(figsize=(10, 5))
    plt.plot(train_loss_history, label='Training Loss')
    plt.title(title)
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def plot_confusion_matrix(conf_matrix, title="Confusion Matrix"):
    plt.figure(figsize=(8, 6))
    sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Racist', 'Caste', 'Offensive', 'Sexist'],
                yticklabels=['Racist', 'Caste', 'Offensive', 'Sexist'])
    plt.title(title)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.show()


In [ ]:

# Plot training curves for both models
plot_training_curve(train_loss_teacher, title="Teacher Model Training Loss Curve")
plot_training_curve(train_loss_student, title="Student Model Training Loss Curve")

# Evaluate the student model on the original test data
accuracy, f1, report, conf = evaluate_model(student_model, test_loader)


### Model Evaluation and Reporting
- **Evaluation Function**: `evaluate_and_report_as_dataframe()` calculates performance metrics for the given model and returns the classification report as a DataFrame.
- **Confusion Matrix**: Plots confusion matrix for better visualization of model performance.



In [ ]:
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Modified function to return classification report as DataFrame
def evaluate_and_report_as_dataframe(model, test_loader, labels):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for inputs, label in test_loader:
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(label.cpu().numpy())

    # Generate classification report as dictionary
    report_dict = classification_report(all_labels, all_preds, target_names=labels, output_dict=True, digits=2)
    report_df = pd.DataFrame(report_dict).transpose()

    # Display classification report as DataFrame
    print("Classification Report:")
    display(report_df)

    # Plot confusion matrix
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(10, 7))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title('Confusion Matrix')
    plt.show()

    return report_df

# Evaluate teacher and student models and display classification reports as DataFrames
labels = ['Racist', 'Caste', 'Offensive', 'Sexist']

print("Results for Test Set(12 samples):")
teacher_report_df = evaluate_and_report_as_dataframe(teacher_model, test_loader, labels)


synthetic_test_dataset = TensorDataset(synthetic_test_embeddings, synthetic_test_labels)
synthetic_test_loader = DataLoader(synthetic_test_dataset, batch_size=8, shuffle=False)

print("Results for Test Set(360 samples):")
student_synthetic_report_df = evaluate_and_report_as_dataframe(student_model, synthetic_test_loader, labels)
